# Youtube Comments Preprocessing

In [ ]:
import pandas as pd 
import numpy as np
import glob
import os
import re
import emoji
import pathlib
import nltk
from nltk.tokenize import sent_tokenize

### Batch Dataset Loading and Initial Exploration

Prior to performing sentiment analysis, topic modeling, and temporal weighting, all CSV files within the selected batch folder were consolidated into a single dataset. This process ensured that data from multiple sources could be analyzed collectively and consistently.

The following operations were performed:

- **Batch Selection and Path Configuration:** The target batch folder was specified and dynamically combined with the base data directory to locate the dataset files.
- **CSV File Discovery:** All CSV files within the selected batch were identified using pattern matching.
- **Dataset Consolidation:** The discovered files were loaded and merged into a single DataFrame (`combined_df`), with indices reset to maintain continuity across records.
- **Validation Check:** A warning message was generated if no CSV files were found in the specified folder.
- **Dataset Profiling:** Summary information about the merged dataset was produced, including:
  - The total number of files merged.
  - The dimensions of the resulting dataset.
  - Column names, data types, and non-null counts.
  - Descriptive statistics for numerical and categorical variables.

This step provided an initial overview of the dataset structure and quality, allowing potential issues such as missing values, inconsistent data types, or anomalous distributions to be identified before proceeding with preprocessing and analysis.

In [ ]:
# 1. Define your base data directory and your dynamic batch name
base_data_dir = '../../data'
batch_name = '08-legal_probe' 

# Combine them to get the exact path for this run
folder_path = os.path.join(base_data_dir, batch_name)

# 2. Grab all the CSV files in that specific batch folder
file_list = glob.glob(os.path.join(folder_path, '*.csv'))

if not file_list:
    print(f"Warning: No CSV files found in '{folder_path}'")
else:
    # 3. Read and concatenate them in one step
    combined_df = pd.concat((pd.read_csv(f) for f in file_list), ignore_index=True)

    # 4. Describe the newly merged dataset
    print(f"--- Dataset Summary for {batch_name} ---")
    print(f"Total files merged: {len(file_list)}")
    print(f"Total rows and columns: {combined_df.shape}\n")
    
    print("--- Column Data Types & Non-Null Counts ---")
    combined_df.info()
    
    print("\n--- Statistical Summary ---")
    print(combined_df.describe(include='all'))


--- Dataset Summary for 07-uaap_ban ---
Total files merged: 4
Total rows and columns: (104, 12)

--- Column Data Types & Non-Null Counts ---
<class 'pandas.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   video_id           104 non-null    str    
 1   video_title        104 non-null    str    
 2   channel_id         0 non-null      float64
 3   comment_id         104 non-null    str    
 4   text               104 non-null    str    
 5   user_handle        104 non-null    str    
 6   user_display_name  104 non-null    str    
 7   like_count         104 non-null    int64  
 8   reply_count        0 non-null      float64
 9   create_time        104 non-null    str    
 10  is_reply           104 non-null    str    
 11  parent_comment_id  104 non-null    str    
dtypes: float64(2), int64(1), str(9)
memory usage: 9.9 KB

--- Statistical Summary ---
      

### Prints the First 5 Rows of the Dataset
To know what columns are important and see sample data 

In [79]:
# Gets the First 5 Rows
print("First 5 Rows: ")
combined_df.head()

First 5 Rows: 


,video_id,video_title,channel_id,comment_id,text,user_handle,user_display_name,like_count,reply_count,create_time,is_reply,parent_comment_id
0,SdvXWSXf8Uc,UAAP bans Ateneo coaching staff present in Aur...,NaN,UgzaQpexVs2os2Y47UV4AaABAg,"Keso? Nanaman? \nPagkatapos ng ""Forthwith inci...",@wilson5377,@wilson5377,2,NaN,2026-06-20T00:00:00+00:00,no,root
1,m2fxJKEPk_M,ICYMI: UAAP bans Ateneo coaching staff present...,NaN,Ugz66a-wQfKCerX6SyN4AaABAg,Yes. Sanction the Coaching Team. Leave the Pla...,@lz23-z8x,@lz23-z8x,0,NaN,2026-06-21T00:00:00+00:00,no,root
2,cvrf9izoNcA,Ateneans walk out to call for justice for Bate...,NaN,Ugwf5vWLqQ0HY7My5S54AaABAg,Nag walk out Ng mga student Ng ateneo de manil...,@eforestry7264,@eforestry7264,0,NaN,2026-06-23T00:00:00+00:00,no,root
3,cvrf9izoNcA,Ateneans walk out to call for justice for Bate...,NaN,UgzJkg5r4MQp4h6nc-54AaABAg,Ang daming eskuwelahan,@ayeshasalic2266,@ayeshasalic2266,0,NaN,2026-06-23T00:00:00+00:00,no,root
4,cvrf9izoNcA,Ateneans walk out to call for justice for Bate...,NaN,Ugy_w1Dzh_AjOGz1wwN4AaABAg,"Salamat sa mga estudyante na ito,pagpalain kay...",@ayeshasalic2266,@ayeshasalic2266,0,NaN,2026-06-23T00:00:00+00:00,no,root


### Data Reduction and Feature Selection

Prior to conducting sentiment analysis, topic modeling, and time-series weighting, the dataset was reduced to include only the attributes relevant to the study. The retained columns were:

- **video_id**: Unique identifier of the associated video.
- **text**: Textual content of the comment for natural language processing tasks.
- **like_count**: Engagement metric used as a potential weighting factor.
- **create_time**: Comment timestamp for temporal analysis.

The original dataframe was overwritten with the filtered dataset to improve computational efficiency and focus subsequent analyses on relevant features. A preview of the resulting dataframe was then displayed for validation.

In [80]:
# Select only the columns needed for sentiment, topic modeling, and time-series weighting
columns_to_keep = ['video_id', 'text', 'like_count', 'create_time']

# Overwrite the dataframe with only the kept columns
combined_df = combined_df[columns_to_keep]

# Display the first few rows to verify the changes
combined_df.head()

,video_id,text,like_count,create_time
0,SdvXWSXf8Uc,"Keso? Nanaman? \nPagkatapos ng ""Forthwith inci...",2,2026-06-20T00:00:00+00:00
1,m2fxJKEPk_M,Yes. Sanction the Coaching Team. Leave the Pla...,0,2026-06-21T00:00:00+00:00
2,cvrf9izoNcA,Nag walk out Ng mga student Ng ateneo de manil...,0,2026-06-23T00:00:00+00:00
3,cvrf9izoNcA,Ang daming eskuwelahan,0,2026-06-23T00:00:00+00:00
4,cvrf9izoNcA,"Salamat sa mga estudyante na ito,pagpalain kay...",0,2026-06-23T00:00:00+00:00


### Text Cleaning Pipeline

A dedicated text cleaning function was defined to standardize comment text prior to analysis. The following operations were applied in sequence:

- **Lowercasing**: All text was converted to lowercase to ensure case-insensitive matching.
- **Emoji Demojization**: Emojis were converted to their textual descriptions (e.g., ":crying_face:") using the `emoji` library, preserving emotional signals present in the comments.
- **URL Removal**: Any HTTP(S) or www URLs were stripped from the text to reduce noise.
- **Character Repetition Normalization**: Repeated characters (e.g., "sooooo") were reduced to two occurrences to normalize informal emphasis while retaining readability.
- **Whitespace Cleanup**: Extra whitespace was collapsed and leading/trailing spaces were trimmed.

Additionally, an `is_shouting` binary feature was derived from the original text by detecting fully uppercased strings longer than five characters, flagging comments that may convey heightened urgency or emotional intensity.

In [81]:
def clean_text_for_crisis(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Convert emojis to text tokens (e.g., 😢 -> :crying_face:)
    text = emoji.demojize(text)
    
    # 3. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # 4. Normalize character repetition (e.g., "sooooo" -> "soo")
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # 5. Final whitespace cleanup
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Add an 'is_shouting' feature to your dataframe before cleaning
combined_df['is_shouting'] = combined_df['text'].apply(lambda x: 1 if isinstance(x, str) and x.isupper() and len(x) > 5 else 0)
combined_df['clean_text'] = combined_df['text'].apply(clean_text_for_crisis)

combined_df.head()

,video_id,text,like_count,create_time,is_shouting,clean_text
0,SdvXWSXf8Uc,"Keso? Nanaman? \nPagkatapos ng ""Forthwith inci...",2,2026-06-20T00:00:00+00:00,0,"keso? nanaman? pagkatapos ng ""forthwith incide..."
1,m2fxJKEPk_M,Yes. Sanction the Coaching Team. Leave the Pla...,0,2026-06-21T00:00:00+00:00,0,yes. sanction the coaching team. leave the pla...
2,cvrf9izoNcA,Nag walk out Ng mga student Ng ateneo de manil...,0,2026-06-23T00:00:00+00:00,0,nag walk out ng mga student ng ateneo de manil...
3,cvrf9izoNcA,Ang daming eskuwelahan,0,2026-06-23T00:00:00+00:00,0,ang daming eskuwelahan
4,cvrf9izoNcA,"Salamat sa mga estudyante na ito,pagpalain kay...",0,2026-06-23T00:00:00+00:00,0,"salamat sa mga estudyante na ito,pagpalain kay..."


### Preprocessing Validation

A sample of the preprocessed comments was inspected to verify the correctness of the cleaning pipeline. The original and cleaned text were compared side-by-side before the original column was dropped, with particular attention to:

- **Emoji conversion**: Confirming that emojis were properly converted to text tokens.
- **Shouting detection**: Verifying that ALL-CAPS comments were flagged correctly.
- **General quality**: Ensuring the cleaned text retained readability and meaning.

In [82]:
# Validation: raw text vs cleaned text, side-by-side
sample = combined_df.sample(10, random_state=42)[['text', 'clean_text', 'is_shouting']]
sample

# Verify emoji demojize preserved meaning
emoji_mask = combined_df['text'].str.contains(r'[😢😡😭😂👍❤️]', na=False)
print(f"\nComments with emojis: {emoji_mask.sum()}")
combined_df.loc[emoji_mask, ['text', 'clean_text']].head(3)


Comments with emojis: 14


,text,clean_text
13,"Oo nman nagpunta ng maynila,para sa pangarap,d...","oo nman nagpunta ng maynila,para sa pangarap,d..."
18,"Baka, baka lang ha dumeretso na naman yan kay ...","baka, baka lang ha dumeretso na naman yan kay ..."
28,5:08 AM here. About to go to early morning mas...,5:08 am here. about to go to early morning mas...


### Sentence Segmentation

Long comments exceeding the 512-token limit of transformer models
(`twitter-xlm-roberta-base-sentiment`) are split into individual
sentences via NLTK's `sent_tokenize`. Each comment retains a
`sentences` list for downstream per-sentence scoring.

In [83]:
# Download tokenizer data if missing
nltk.download('punkt_tab', quiet=True)

# Add a column with list of sentences per comment
combined_df['sentences'] = combined_df['clean_text'].apply(sent_tokenize)

# Quick stats
long_mask = combined_df['clean_text'].str.len() > 500
print(f"Comments exceeding ~500 chars: {long_mask.sum()} ({long_mask.mean():.1%})")
combined_df.head()

Comments exceeding ~500 chars: 3 (2.9%)


,video_id,text,like_count,create_time,is_shouting,clean_text,sentences
0,SdvXWSXf8Uc,"Keso? Nanaman? \nPagkatapos ng ""Forthwith inci...",2,2026-06-20T00:00:00+00:00,0,"keso? nanaman? pagkatapos ng ""forthwith incide...","[keso?, nanaman?, pagkatapos ng ""forthwith inc..."
1,m2fxJKEPk_M,Yes. Sanction the Coaching Team. Leave the Pla...,0,2026-06-21T00:00:00+00:00,0,yes. sanction the coaching team. leave the pla...,"[yes., sanction the coaching team., leave the ..."
2,cvrf9izoNcA,Nag walk out Ng mga student Ng ateneo de manil...,0,2026-06-23T00:00:00+00:00,0,nag walk out ng mga student ng ateneo de manil...,[nag walk out ng mga student ng ateneo de mani...
3,cvrf9izoNcA,Ang daming eskuwelahan,0,2026-06-23T00:00:00+00:00,0,ang daming eskuwelahan,[ang daming eskuwelahan]
4,cvrf9izoNcA,"Salamat sa mga estudyante na ito,pagpalain kay...",0,2026-06-23T00:00:00+00:00,0,"salamat sa mga estudyante na ito,pagpalain kay...","[salamat sa mga estudyante na ito,pagpalain ka..."


### Dataset Summary and Export

Before exporting, the following final preprocessing steps were performed:

- **Datetime Conversion**: The `create_time` column was parsed into `datetime64` format to enable temporal analysis.
- **Deduplication**: Duplicate comments based on identical cleaned text were removed, retaining only the first occurrence.

The final preprocessed dataset was summarized and exported to the `output/` directory as CSV for downstream consumption.

In [84]:
# Convert create_time to datetime
combined_df['create_time'] = pd.to_datetime(combined_df['create_time'])

# Drop exact duplicate texts
before_dedup = len(combined_df)
combined_df = combined_df.drop_duplicates(subset='clean_text', keep='first')
print(f"Dropped {before_dedup - len(combined_df)} duplicate comments")

# Summary statistics
print("\n--- Preprocessed Dataset Summary ---")
print(f"Total comments: {len(combined_df)}")
print(f"Unique videos: {combined_df['video_id'].nunique()}")
print(f"Date range: {combined_df['create_time'].min()} to {combined_df['create_time'].max()}")
print(f"Shouting comments (%): {(combined_df['is_shouting'].mean() * 100):.1f}%")
print(f"Comments with emojis: {emoji_mask.sum()} ({(emoji_mask.mean() * 100):.1f}%)")

clean_len = combined_df['clean_text'].str.len()
print(f"\n--- Text Length Stats ---")
print(f"Mean: {clean_len.mean():.0f} chars, median: {clean_len.median():.0f}")

vocab = set(' '.join(combined_df['clean_text'].dropna()).split())
print(f"Vocabulary size: {len(vocab)}")

nulls = combined_df.isnull().sum()
if nulls.any():
    print(f"\n--- Null Values ---")
    print(nulls[nulls > 0])
else:
    print("\nNo null values in any column")


Dropped 4 duplicate comments

--- Preprocessed Dataset Summary ---
Total comments: 100
Unique videos: 4
Date range: 2026-06-20 00:00:00+00:00 to 2026-06-23 21:00:00+00:00
Shouting comments (%): 5.0%
Comments with emojis: 14 (13.5%)

--- Text Length Stats ---
Mean: 128 chars, median: 88
Vocabulary size: 1014

No null values in any column


### Export Processed Dataset

The preprocessed dataset was exported to the `output/` directory as CSV for downstream consumption.

In [85]:
output_dir = pathlib.Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / f'{batch_name}.csv'

combined_df = combined_df.drop(columns=['text'])
combined_df = combined_df.rename(columns={'clean_text': 'text'})
combined_df = combined_df[['video_id', 'text', 'like_count', 'create_time', 'is_shouting', 'sentences']]
combined_df.to_csv(csv_path, index=False)

print(f"Exported: {csv_path.resolve()}")

Exported: /home/mbchavez/Coding/Research/crisis-management-analysis/01-preprocessing/output/07-uaap_ban.csv
